# 05 - Análise da Busca de Hiperparâmetros (Optuna / TabM)

Gera dois gráficos a partir de `results/tuning_trials_kaggle.csv`:
1. **Curvas de convergência** — melhor AUC acumulado por trial, para cada dataset
2. **Análise de sensibilidade** — distribuição de AUC CV por valor de `num_emb_type`, `tabm_k` e `n_epochs`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

trials = pd.read_csv('../results/tuning_trials_kaggle.csv')
figures = Path('../results/figures')
figures.mkdir(exist_ok=True)

print(f"Trials carregados: {len(trials)} linhas")
print(f"Datasets: {sorted(trials['dataset'].unique())}")
trials.head()

## 1. Curvas de Convergência — Busca Optuna

In [ ]:
DATASET_META_REGIME = {
    'blood-transfusion-service-center': 'small',
    'diabetes': 'small',
    'balance-scale': 'small',
    'credit-g': 'medium',
    'phoneme': 'medium',
    'cmc': 'medium',
    'adult': 'large',
    'bank-marketing': 'large',
    'connect-4': 'large',
}
DATASET_SHORT = {
    'blood-transfusion-service-center': 'blood-transf.',
    'diabetes': 'diabetes',
    'balance-scale': 'balance-scale',
    'credit-g': 'credit-g',
    'phoneme': 'phoneme',
    'cmc': 'cmc',
    'adult': 'adult',
    'bank-marketing': 'bank-mktg',
    'connect-4': 'connect-4',
}
REGIME_ORDER = ['small', 'medium', 'large']
color_regime = {'small': '#2563EB', 'medium': '#16A34A', 'large': '#DC2626'}

datasets_ordered = sorted(
    DATASET_META_REGIME.keys(),
    key=lambda d: (REGIME_ORDER.index(DATASET_META_REGIME[d]), d)
)

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
axes = axes.flatten()

for idx, dataset in enumerate(datasets_ordered):
    ax = axes[idx]
    sub = trials[trials['dataset'] == dataset].sort_values('trial').copy()
    regime = DATASET_META_REGIME[dataset]
    color = color_regime[regime]
    if not sub.empty:
        sub['best_so_far'] = sub['auc_cv'].cummax()
        ax.plot(sub['trial'], sub['auc_cv'], 'o', color=color, alpha=0.4, markersize=4, zorder=2)
        ax.plot(sub['trial'], sub['best_so_far'], '-', color=color, linewidth=2, zorder=3,
                label=f"best={sub['best_so_far'].iloc[-1]:.4f}")
        ax.legend(fontsize=7, loc='lower right')
    ax.set_title(f"{DATASET_SHORT[dataset]} ({regime})", fontsize=9, fontweight='bold')
    ax.set_xlabel('Trial', fontsize=8)
    ax.set_ylabel('AUC CV', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

patches = [mpatches.Patch(color=c, label=r) for r, c in color_regime.items()]
fig.legend(handles=patches, loc='upper center', ncol=3, fontsize=9,
           title='Regime', bbox_to_anchor=(0.5, 1.01))
fig.suptitle('Curvas de Convergência — Busca Optuna (TabM)', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(figures / 'optuna_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Salvo em results/figures/optuna_convergence.png")

## 2. Análise de Sensibilidade dos Hiperparâmetros

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# num_emb_type
ax = axes[0]
emb_order = ['none', 'lt', 'pbld', 'plr']
bp = ax.boxplot(
    [trials[trials['num_emb_type'] == e]['auc_cv'].values for e in emb_order],
    tick_labels=emb_order, patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch, c in zip(bp['boxes'], ['#94A3B8', '#60A5FA', '#34D399', '#FBBF24']):
    patch.set_facecolor(c); patch.set_alpha(0.8)
ax.set_title('num_emb_type', fontsize=11, fontweight='bold')
ax.set_ylabel('AUC CV', fontsize=10)
ax.set_xlabel('Tipo de Embedding', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# tabm_k
ax = axes[1]
k_vals = sorted(trials['tabm_k'].unique())
bp = ax.boxplot(
    [trials[trials['tabm_k'] == k]['auc_cv'].values for k in k_vals],
    tick_labels=k_vals, patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch in bp['boxes']:
    patch.set_facecolor('#60A5FA'); patch.set_alpha(0.7)
ax.set_title('tabm_k (nº de cabeças)', fontsize=11, fontweight='bold')
ax.set_ylabel('AUC CV', fontsize=10)
ax.set_xlabel('k', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# n_epochs
ax = axes[2]
ep_vals = sorted(trials['n_epochs'].unique())
bp = ax.boxplot(
    [trials[trials['n_epochs'] == e]['auc_cv'].values for e in ep_vals],
    tick_labels=ep_vals, patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch in bp['boxes']:
    patch.set_facecolor('#34D399'); patch.set_alpha(0.7)
ax.set_title('n_epochs', fontsize=11, fontweight='bold')
ax.set_ylabel('AUC CV', fontsize=10)
ax.set_xlabel('Épocas', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Análise de Sensibilidade dos Hiperparâmetros (TabM)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(figures / 'sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Salvo em results/figures/sensitivity_analysis.png")